In [0]:
%python
customers_df2=spark.read.table('retails_project.silver.customers_silver')
customers_df2.display()

orders_df2=spark.read.table('retails_project.silver.orders_silver')
orders_df2.display()

In [0]:
%python
dim_customer=customers_df2.select('customer_id','customer_name','email','city','state','loyalty_points','status')
dim_customer.display()

dim_customer.write.format("delta").mode('overwrite').saveAsTable('retails_project.gold.gold_dim_customer')

In [0]:
%python
from pyspark.sql.functions import year , month , dayofmonth

fact_sales=orders_df2.join(
    customers_df2, 
    orders_df2.customer_id==customers_df2.customer_id, 
    'left'
).select(
    orders_df2.order_id,
    orders_df2.order_date,
    orders_df2.customer_id,
    customers_df2.customer_name,
    orders_df2.product_id,
    orders_df2.product_name,
    orders_df2.category,
    orders_df2.quantity,
    orders_df2.unit_price,
    orders_df2.discount,
    orders_df2.total_amount,
    customers_df2.city,
    customers_df2.state,
    customers_df2.gender,
    customers_df2.loyalty_points
)

order_enriched=fact_sales.\
    withColumn("Year", year("order_date")).\
    withColumn("Month", month("order_date")).\
    withColumn("Day", dayofmonth("order_date"))

order_enriched.display()

In [0]:
%python
from pyspark.sql.functions import *

gold_order_df=order_enriched.groupBy(
    "product_id",
    "product_name",
    "customer_id",
    "customer_name",
    "category",
    "Year",
    "Month",
    "order_date",
    "gender",
    "city",
    "state",
    "loyalty_points"
).agg(
    sum("total_amount").alias("Total_Revenue"),
    sum("quantity").alias("Total_Quantity"),
    countDistinct("order_id").alias("Total_Orders"),
    avg("total_amount").alias("Avg_Revenue"),
    avg("quantity").alias("Avg_Quantity"),
    avg('loyalty_points').alias('Avg_Loyalty'),
    min('unit_price').alias("Min_Price"),
    max("unit_price").alias("Max_Price")
)
display(gold_order_df)

gold_order_df.write.format("delta").mode('overwrite').saveAsTable('retails_project.gold.gold_fact_sales')